# Lab 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishwar97/geog-510/blob/main/book/labs/lab_05.ipynb)

## Exercise 1: Calculating Distances with Functions

- Define a function `calculate_distance` that takes two geographic coordinates (latitude and longitude) and returns the distance between them using the Haversine formula.
- Use this function to calculate the distance between multiple pairs of coordinates.

In [ ]:
from math import sin, radians, cos, sqrt, atan2
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6371 # Radius of Earth in kms
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = (
        sin(dlat / 2) ** 2
        + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    )
    c = 2 * atan2(sqrt(a),sqrt(1-a))
    distance = R * c
    return distance

In [ ]:
coordinates = [
    (-34.6789, 75.8985),
    (19.7475, 79.1412),
    (-23.6923, 35.1890),
    (19.7842, 56.2343),
]
# Calculating distance between first two points
lat1, lon1 = coordinates[0]
lat2, lon2 = coordinates[1]
distance_km = calculate_distance(lat1,lon1,lat2,lon2)
print(f"{distance_km:.2f}")

## Exercise 2: Batch Distance Calculation

- Create a function `batch_distance_calculation` that accepts a list of coordinate pairs and returns a list of distances between consecutive pairs.
- Test the function with a list of coordinates representing several cities.

In [ ]:
def batch_distance_calculation(coordinates_list):
    distances = []
    for i in range(len(coordinates_list) - 1):
        lat1, lon1 = coordinates_list[i]
        lat2, lon2 = coordinates_list[i + 1]
        distance_km = round(calculate_distance(lat1, lon1, lat2, lon2),2)
        distances.append(distance_km)
    return distances

# Test
distances = batch_distance_calculation(coordinates)
print(f"Distances bet consecutive points in list (kms): {distances}")

## Exercise 3: Creating and Using a Point Class

- Define a `Point` class to represent a geographic point with attributes `latitude`, `longitude`, and `name`.
- Add a method `distance_to` that calculates the distance from one point to another.
- Instantiate several `Point` objects and calculate the distance between them.

In [ ]:
class Point:
    def __init__(self, latitude, longitude, name=None):
        self.latitude = float(latitude)
        self.longitude = float(longitude)
        self.name = name

    def __str__(self):
        return f"{self.name or 'Point'} ({self.latitude}, {self.longitude})"
    
    def distance_to(self, otherpoint):
        return calculate_distance(self.latitude, self.longitude, otherpoint.latitude, otherpoint.longitude)

# Example    
point1 = Point(28.644650910589, 77.17330317895471, "New Delhi")
point2 = Point(19.025425076920786, 72.87340293777716, "Mumbai")
print(
    f"Distance from {point1.name} to {point2.name}: {point1.distance_to(point2):.2f} km"
)  


## Exercise 4: Reading and Writing Files

- Write a function `read_coordinates` that reads a file containing a list of coordinates (latitude, longitude) and returns them as a list of tuples.
- Write another function `write_coordinates` that takes a list of coordinates and writes them to a new file.
- Ensure that both functions handle exceptions, such as missing files or improperly formatted data.

In [ ]:
# Create a sample coordinates.txt file
sample_data = """35.6895,139.6917
34.0522,-118.2437
51.5074,-0.1278
-33.8688,151.2093
48.8566,2.3522"""

output_file = "coordinates.txt"

try:
    with open(output_file, "w") as file:
        file.write(sample_data)
    print(f"Sample file '{output_file}' has been created successfully.")
except Exception as e:
    print(f"An error occurred while creating the file: {e}")

In [ ]:
def read_coordinates(in_file):
    '''
    Args:
    in_file: input path to txt file with coordinates stored in "latitude, longtide" format
    Returns:
    List: list of tuples with format (latitude, longitde)
    '''
    try:
        with open(in_file, 'r') as infile:
            coordinates = infile.readlines()
    except Exception as e:
        print(f"Error : {e}")
    coords_list = []
    for line in coordinates:
        try:
            lat, lon = line.strip().split(',')
            coords_list.append((float(lat),float(lon)))
        except:
            pass
    return coords_list

In [ ]:
coord_list = read_coordinates('coordinates.txt')
print(coord_list)

In [ ]:
def write_coordinates(coords_list, output_file):
    try:
        with open(output_file,'w') as outfile:
            for i in range(0,len(coords_list)):
                lat, long = coords_list[i]
                outfile.write(f'Latitude: {lat}, Longitude: {long}'+'\n')
    except Exception as e:
        print(e)
    return None

# Example usage
write_coordinates(coord_list,'coordinates_new.txt')

## Exercise 5: Processing Coordinates from a File

- Create a function that reads coordinates from a file and uses the `Point` class to create `Point` objects.
- Calculate the distance between each consecutive pair of points and write the results to a new file.
- Ensure the function handles file-related exceptions and gracefully handles improperly formatted lines.

In [ ]:
def parse_coordinates(line):
    try:
        if "," in line:
            lat, long = line.strip().split(",")
        elif ";" in line:
            lat, long = line.strip().split(";")
        elif "  " in line:
            lat, long = line.strip().split("  ")
        return lat, long
    except Exception as e:
        print(f'Error: {e}. Problem reading line: {line.strip()}')
        return None
    
def process_geospatial_file(in_file,out_file):
    try:
        distances = []
        with open(in_file,'r') as infile:
            lines = infile.readlines()
            for i in range(0, len(lines)-1):
                point1 = Point(parse_coordinates(lines[i])[0],parse_coordinates(lines[i])[1],f"Point_{i+1}")
                point2 = Point(parse_coordinates(lines[i+1])[0],parse_coordinates(lines[i+1])[1],f"Point_{i+2}")
                distance = point2.distance_to(point1)
                distance_line = f"Distance between {point1.name} and {point2.name}: {distance:.2f} kms"
                distances.append(distance_line)
                # print(distance_line)
        
        if '.txt' in out_file:
            try:
                 with open(out_file,'w') as outfile:
                      for dline in distances:
                        outfile.writelines(f'{dline}\n')
            except Exception as e:
                print(f"Error occured while writing file: {e}")
        else:
            raise ValueError("Unsupported Output File Format!")
    except FileNotFoundError:
        print(f"Error: Input file {in_file} not found!")
    except Exception as e:
        print(f"Unexpected error occured while processing file: {e}")
    return None

In [ ]:
# Example Usage
process_geospatial_file("coordinates.txt","distances.txt")

## Exercise 6: Exception Handling in Data Processing

- Modify the `batch_distance_calculation` function to handle exceptions that might occur during the calculation, such as invalid coordinates.
- Ensure the function skips invalid data and continues processing the remaining data.

In [ ]:
coordinates = [
    (-34.6789, 1,75.8985),
    (19.7475, 79.1412),
    (-23.6923, 35.1890),
    (19.7842, 56.2343),
    (19.7842, 1.34, 55.2343)
]
def batch_distance_calculation(coordinates_list):
    distances = []
    for i in range(len(coordinates_list)-1):
        try:
            lat1, lon1 = coordinates_list[i]
            lat2, lon2 = coordinates_list[i+1]
            distance_km = round(calculate_distance(lat1, lon1, lat2, lon2),2)
            dLine = f"Distance between {coordinates_list[i]} and {coordinates_list[i+1]}: {distance_km} kms"
            print(dLine)
            distances.append(distance_km)
        except Exception as e:
            print(f'Error occured for points {coordinates_list[i], coordinates_list[i+1]}: {e}')
            pass
    return distances

# Test
distances = batch_distance_calculation(coordinates)
print(f"Distances between consecutive points (kms): {distances}")

## Exercise 7: NumPy Array Operations and Geospatial Coordinates

In this exercise, you will work with NumPy arrays representing geospatial coordinates (latitude and longitude) and perform basic array operations.

1. Create a 2D NumPy array containing the latitude and longitude of the following cities: Tokyo (35.6895, 139.6917), New York (40.7128, -74.0060), London (51.5074, -0.1278), and Paris (48.8566, 2.3522).
2. Convert the latitude and longitude values from degrees to radians using np.radians().
3. Calculate the element-wise difference between Tokyo and the other cities' latitude and longitude in radians.

In [ ]:
import numpy as np
# 1. Create 2D np array for coordinates 
np_array = np.array([
    [35.6895, 139.6917],
    [40.7128, -74.0060],
    [51.5074, -0.1278], 
    [48.8566, 2.3522]
])
# 2. Convert degrees to radians using np.radian()
coords_radians = np.radians(np_array)
print(f"Coordinates in radians:\n{coords_radians}")

# 3. Element wise difference between Tokyo and other city coordinates in radians
np_diff = np.radians(np_array[1:]) - np.radians(np_array[0])
print("Difference in radian:\n",np_diff)


## Exercise 8: Pandas DataFrame Operations with Geospatial Data

In this exercise, you'll use Pandas to load and manipulate a dataset containing city population data, and then calculate and visualize statistics.

1. Load the world cities dataset from this URL using Pandas: https://github.com/opengeos/datasets/releases/download/world/world_cities.csv
2. Display the first 5 rows and check for missing values.
3. Filter the dataset to only include cities with a population greater than 1 million.
4. Group the cities by their country and calculate the total population for each country.
5. Sort the cities by population in descending order and display the top 10 cities.

In [ ]:
import pandas as pd

# 1. Load csv from github
cities = pd.read_csv("https://github.com/opengeos/datasets/releases/download/world/world_cities.csv")

# 2. Print first 5 rows and check for missing values
print(cities.head(5))
print(cities.isnull().sum())

# 3. Filter cities with population > 1e6
populated_cities = cities[cities['population']>1e6]
# print(populated_cities.head(2))

# 4. Group cities based on countries and calculate total population for each country
country_agg = cities.groupby('country')['population'].sum()
# print(country_agg.head(5))

# 5. Sort cities in descending order and display 10
print(cities.sort_values(by="population",ascending=False).head(10))

## Exercise 9: Creating and Manipulating GeoDataFrames with GeoPandas

This exercise focuses on creating and manipulating GeoDataFrames, performing spatial operations, and visualizing the data.

1. Load the New York City building dataset from the GeoJSON file using GeoPandas: https://github.com/opengeos/datasets/releases/download/places/nyc_buildings.geojson
2. Create a plot of the building footprints and color them based on the building height (use the `height_MS` column).
3. Create an interactive map of the building footprints and color them based on the building height (use the `height_MS` column).
4. Calculate the average building height (use the `height_MS` column).
5. Select buildings with a height greater than the average height.
6. Save the GeoDataFrame to a new GeoJSON file.

In [ ]:
import geopandas as gpd
import requests
url = (
    "https://github.com/opengeos/datasets/releases/download/places/nyc_buildings.geojson"
)

# 1. Load New York Buildings dataset
nyc_buildings = gpd.read_file(url)

In [ ]:
# 2. Building footprint plot using height_MS
nyc_buildings.plot("height_MS", legend=True,figsize=(10,6),cmap="magma")

In [ ]:
# 3. Interactive map with building color coded using height_MS column
nyc_buildings.explore("height_MS",cmap="magma")

In [ ]:
# 4. Average building height
avg_height = nyc_buildings['height_MS'].mean()

# 5. Filter building based on avg height
tall_buildings = nyc_buildings[nyc_buildings['height_MS']>avg_height]

# 6. Save geodataframe to new geosjon file
tall_buildings.to_file("buildings_subset_nyc.geojson")

## Exercise 10: Combining NumPy, Pandas, and GeoPandas

This exercise requires you to combine the power of NumPy, Pandas, and GeoPandas to analyze and visualize spatial data.

1. Use Pandas to load the world cities dataset from this URL: https://github.com/opengeos/datasets/releases/download/world/world_cities.csv
2. Filter the dataset to include only cities with latitude values between -40 and 60 (i.e., cities located in the Northern Hemisphere or near the equator).
3. Create a GeoDataFrame from the filtered dataset by converting the latitude and longitude into geometries.
4. Reproject the GeoDataFrame to the Mercator projection (EPSG:3857).
5. Calculate the distance (in meters) between each city and the city of Paris.
6. Plot the cities on a world map, coloring the points by their distance from Paris.

In [ ]:
cities = pd.read_csv("https://github.com/opengeos/datasets/releases/download/world/world_cities.csv")

In [ ]:
# 2. Filter cities based on latitude range
cities_filtered = cities[(cities['latitude'] > - 40) | (cities['latitude'] < 60)]
# print(cities_filtered.head(2))

# 3. Create gdf from cities_filtered by converting lat, long to geometries
geom = gpd.points_from_xy(cities_filtered['longitude'],cities_filtered['latitude'])
cities_gdf = gpd.GeoDataFrame(cities_filtered, geometry=geom, crs="EPSG:4326")
print(cities_gdf.head(5))


In [ ]:
# 4. Reproject cities_gdf from 4326 to 3857
cities_gdf.to_crs(crs="EPSG:3857",inplace=True)
# print(cities_gdf.crs)

# 5. Calculate distance of cities from Paris
# cities_gdf.to_crs(crs="EPSG:4326",inplace=True)
paris = cities_gdf[cities_gdf['name']=="Paris"]['geometry'].iloc[0]
cities_gdf["distance_to_paris_m"] = cities_gdf.distance(paris)
print(cities_gdf["distance_to_paris_m"].head(5))

In [ ]:
cities_gdf.to_crs(crs="EPSG:4326")
cities_gdf.plot(column="distance_to_paris_m",legend=True,cmap="OrRd")

## Submission Requirements

Complete the exercises above and and upload the notebook to your GitHub repository. Make sure the notebook has a Colab badge at the top so that it can be easily opened in Google Colab. Submit the URL of the notebook to Canvas.